# Fig. 7：CRC / PDAC 上游单细胞分析

对应 `Fig7_JUN_only_symmetric.pdf`，与 `fig7-1_cross_cancer_jun_reference_and_tnk_associations.ipynb` 配套。

本文件补入清理、上皮重聚类、ccRCC 参考准备与 JUN 匹配、独立 CM 拟合、样本级关联及绘图源表导出。实际科学代码保存在同目录 `fig7_upstream_sources/`，原脚本完整保留，来源和 SHA256 见 `SOURCE_MANIFEST.json`。入口执行本地副本，但将 `__file__` 锚定原位置，保留原输入、输出及参数。

起点是已有的表达/注释 h5ad、样本 metadata 和背景 atlas，而非 FASTQ/BAM。CRC 新 T/NK 标签及两癌种背景 atlas 的生成源码也已收录；历史全 atlas/CM 分支只用于追溯背景，不替代当前 calibrated 分支。当前 Fig. 7 只分析单细胞 JUN-reference / T/NK 模块关系，不启动旧空间、治疗或临床分支。

这次整理未运行重聚类、NMF 或相关性分析，也未改动原图和统计结果。

## 0. 设置与只读检查

默认 `RUN_ANALYSES = False`，仅显示阶段；真正重建时须显式改为 `True`。执行将写入原分析目录，并可能覆盖同名缓存/结果，先备份。CRC 的原始拟合/聚类函数保留其缓存复用行为，不会为了“从头”自动删除旧缓存。

CRC 与 PDAC 沿用各自的原环境，不能假定同一 kernel 能保持两者数值重现性。尤其 PDAC 使用记录中的 Harmony 0.0.10；软件版本副本在各自 calibrated 子目录内。切换环境后重新运行设置单元，再执行相应癌种部分。

In [ ]:
from pathlib import Path
import importlib.util
import sys

HERE = Path.cwd()
if not (HERE / 'fig7_upstream_sources/runner.py').exists():
    HERE = Path('/mnt/disk18t/lr_xcy/riku/kirc_205/main_codes/fig7')
spec = importlib.util.spec_from_file_location('fig7_upstream_runner', HERE / 'fig7_upstream_sources/runner.py')
upstream = importlib.util.module_from_spec(spec)
spec.loader.exec_module(upstream)
RUN_ANALYSES = False
checks = upstream.preflight()
for row in checks:
    if not row['exists_and_valid']:
        print('MISSING / CHANGED:', row['path'])
print(f"{sum(r['exists_and_valid'] for r in checks)}/{len(checks)} source/input checks passed")


## 1. CRC：必要的更早输入生成代码

已有输入可直接跳到第 2 部分。以下仅在需要重建对应输入时单独执行：旧版 reviewed mask → 全 atlas 表达整合和粗谱系 → 非上皮谱系重聚类（包含当前 11 个表达型 T/NK 群）→ ccRCC 冻结参考。

`full_atlas_reintegration_20261107` 的旧上皮/CM/关联源码亦随包保留，用于追溯背景表 `all_cell_final_labels.csv.gz`；它们不是本次 Fig. 7 的主上皮/CM 版本，不自动运行。

当前 CRC `01_prepare.py` 原脚本仍读取 `cm_epi_redesign/spatial/panel_genes.csv` 并导出 panel 投影矩阵。这只是保留其历史输入依赖；最终 JUN-only 图不使用空间分析。其生成代码在 archived `cm_epi_redesign/04_spatial.py:prepare`，现有文件保留即可。

In [ ]:
# 仅按需取消注释；当前已有这些输入。
# upstream.run_stage('CRC.reviewed_clean_mask', execute=RUN_ANALYSES)
# upstream.run_stage('CRC.background_atlas', execute=RUN_ANALYSES)
# upstream.run_stage('CRC.non_epithelial_reclustering', execute=RUN_ANALYSES)
# upstream.run_stage('CRC.freeze_reference', execute=RUN_ANALYSES)


## 2. CRC：当前 calibrated 单细胞主分析

原始顺序：`01_prepare.py` 清理/表达导出 → `02_CM.py` 独立 CM 拟合与选择 → `03_expression_clustering.py` 肿瘤上皮表达聚类 → `04_reference_and_similarity.py` 参考匹配/marker/bootstrap/LOSO → `05_singlecell.py` 样本/患者键及阈值敏感性关联。

主版本：HVG3000、Leiden 0.4，15 个上皮 cluster；9 个满足支持阈值，JUN-like 为 c08。当前 CM 为 14 模块中的 CRCcal_CM08，不复用历史 CM 结果。主图展示 ≥30 上皮、≥50 非上皮的肿瘤样本，JUN fraction 与 raw NNLS 活性的 nominal Spearman P。VIM/balance 及其他敏感性结果保留在原始完整输出，但不替换主图结果。

In [ ]:
upstream.run_stage('CRC.prepare', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('CRC.CM', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('CRC.epithelial_clustering', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('CRC.reference_matching', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('CRC.associations', execute=RUN_ANALYSES)


## 3. PDAC：已有 cleaned ductal 输入的生产代码

`cm_epi_redesign/scripts/02_epithelial_sweep.py` 是当前 `cleaned_tumor_ductal.h5ad`、清理 metadata 及历史对照表的原始生产脚本；它包含旧的程序评分与聚类扫参，仅用于恢复这些前置文件，旧标签不作为当前 c7 的标签。文件已完整收录；已有输入无需执行本单元。

In [ ]:
# upstream.run_stage('PDAC.archived_ductal_preparation', execute=RUN_ANALYSES)


## 4. PDAC：当前 calibrated 单细胞主分析

独立冻结 ccRCC 参考 → 审核原非上皮表达/清理证据 → CM 校准 → 上皮表达扫参 → 参考匹配 → 单细胞关联。

当前主版本为 `A_HVG3000_sample_r0.4`：12 个 cluster、9 个满足支持阈值，JUN-like 为 c7；独立 CM 为 11 模块中的 calCM02。不能用旧历史重放的 13 群版本替换。

`04_three_modalities.py` 同时含临床/空间分析，下面只装载并调用其原始 `singlecell(pair, cm)` 函数，函数体未改，不执行该文件的 main、clinical 或 spatial。

In [ ]:
upstream.run_stage('PDAC.freeze_reference', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('PDAC.cleanup_evidence', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('PDAC.CM', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('PDAC.epithelial_clustering', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('PDAC.reference_matching', execute=RUN_ANALYSES)


In [ ]:
upstream.run_stage('PDAC.associations', execute=RUN_ANALYSES)


## 5. 生成 Fig. 7 所需源表

先提取旧 finalization/report 中真正用到的单细胞表和 normal-reference 标签，避免为了得到这些表而运行整个临床/空间报告。然后用原 atlas 导出函数将当前标签叠加到已保存的背景坐标；不重新拟合 UMAP。

最后调用原相似性源表脚本，并按原绘图脚本的列映射、阈值、窗口和 loading 定义导出 JUN-only 绘图输入。不重新选 cluster/CM，不重新校正 P，不用旧 ccRCC/临床/空间结果替代当前数据。该适配层的代码在 `runner.py`，与原脚本副本明确区分。共享表内已有的其他癌种行会保留。

In [ ]:
upstream.export_pdac_singlecell_display_inputs(execute=RUN_ANALYSES)
upstream.run_stage('CRC.atlas_plot_data', execute=RUN_ANALYSES)
upstream.run_stage('PDAC.atlas_plot_data', execute=RUN_ANALYSES)
upstream.export_figure_inputs(execute=RUN_ANALYSES)


## 6. 绘图

完成源表核对后运行 `fig7-1_cross_cancer_jun_reference_and_tnk_associations.ipynb`。A/B 沿用背景 atlas；C/D/E/F 使用已确认的新轴线样式。不在本文件提前绘图。

当前保存结果校验锚点：CRC 127 个肿瘤样本，ρ = 0.30617662354682107，P = 0.0004638927111977865；PDAC 74 个，ρ = 0.4375224046576715，P = 9.697117975595206e-05。这些是核对锚点，不能在重跑结果不同的情况下强写回旧值。

边界与详细来源见 `fig7_upstream_sources/README.md`、`SOURCE_MANIFEST.json`。